# 面试题：工作记忆、情景记忆和长期记忆怎样分层？

工作记忆服务当前任务，情景记忆保存可检索事件，长期记忆只保存已验证的稳定偏好。三层使用不同 TTL、写入条件和 ACL；模型推测不能直接进入长期层。

## 真实案例

售后 Agent 面对当前订单、历史退款、语言偏好、过期地址、模型猜测和确认偏好六条记录。

## 基线

基线把所有记录平铺为同等事实。

## 结果解读

分层读取会过滤过期或未验证条目。

## 失败案例

模型猜测的 VIP 身份不能进入长期记忆。

In [1]:
records = [{'key':'order','value':'A12退款中','layer':'work','fresh':True,'verified':True}, {'key':'refund','value':'上次退款30','layer':'episode','fresh':True,'verified':True}, {'key':'lang','value':'中文','layer':'long','fresh':True,'verified':True}, {'key':'address','value':'上海','layer':'episode','fresh':False,'verified':True}, {'key':'vip','value':'是','layer':'long','fresh':True,'verified':False}, {'key':'lang','value':'中文','layer':'long','fresh':True,'verified':True}]  # 构造六条分层记忆记录。
print('记忆输入:', records)  # 输出原始层级、时效和验证字段。
print('教学说明：用户偏好需要确认与撤销路径，事实仍应引用权威资源 ID。')  # 说明记忆边界。

记忆输入: [{'key': 'order', 'value': 'A12退款中', 'layer': 'work', 'fresh': True, 'verified': True}, {'key': 'refund', 'value': '上次退款30', 'layer': 'episode', 'fresh': True, 'verified': True}, {'key': 'lang', 'value': '中文', 'layer': 'long', 'fresh': True, 'verified': True}, {'key': 'address', 'value': '上海', 'layer': 'episode', 'fresh': False, 'verified': True}, {'key': 'vip', 'value': '是', 'layer': 'long', 'fresh': True, 'verified': False}, {'key': 'lang', 'value': '中文', 'layer': 'long', 'fresh': True, 'verified': True}]
教学说明：用户偏好需要确认与撤销路径，事实仍应引用权威资源 ID。


In [2]:
flat = {row['key']:row['value'] for row in records}  # 构造丢失层级与验证信息的平铺基线。
print('平铺基线:', flat)  # 输出所有内容看似同等可信的视图。
print('基线风险：过期地址和模型猜测会成为后续 prompt 的事实。')  # 解释简单拼接的污染。

平铺基线: {'order': 'A12退款中', 'refund': '上次退款30', 'lang': '中文', 'address': '上海', 'vip': '是'}
基线风险：过期地址和模型猜测会成为后续 prompt 的事实。


In [3]:
priority = {'work':3,'episode':2,'long':1}  # 定义当前任务优先于历史和稳定偏好的读取顺序。
def retrieve(rows):  # 定义分层记忆的可用记录选择函数。
    usable = [row for row in rows if row['fresh'] and row['verified']]  # 过滤过期和未经验证的条目。
    view = {}  # 初始化当前动作可消费的记忆视图。
    for row in sorted(usable, key=lambda item:priority[item['layer']], reverse=True):  # 按层级优先级处理可信条目。
        view.setdefault(row['key'], row)  # 每个键只保留最高优先级的可信来源。
    return view  # 返回带层级元数据的安全视图。

In [4]:
view = retrieve(records)  # 执行分层读取。
print('key | value | layer')  # 输出可用记忆表标题。
for key, row in view.items():  # 遍历实际可进入当前任务的记忆。
    print(key, row['value'], row['layer'])  # 输出值与来源层级。
print('过期地址存在:', 'address' in view, '，未验证 VIP 存在:', 'vip' in view)  # 输出过滤中间结果。

key | value | layer
order A12退款中 work
refund 上次退款30 episode
lang 中文 long
过期地址存在: False ，未验证 VIP 存在: False


In [5]:
wrong = flat['vip']  # 读取平铺基线错误保留的模型猜测。
fixed = 'vip' in view  # 读取分层门禁对该猜测的处理。
print('失败案例：平铺 VIP=', wrong, '，分层可用=', fixed)  # 展示未验证长期记忆被拒绝。
print('生产差距：需租户 ACL、TTL、召回、去重、用户删除和源版本追踪。')  # 说明生产记忆系统。

失败案例：平铺 VIP= 是 ，分层可用= False
生产差距：需租户 ACL、TTL、召回、去重、用户删除和源版本追踪。


In [6]:
assert view['order']['layer'] == 'work'  # 验证当前订单来自工作记忆。
assert 'address' not in view  # 验证过期情景记忆不被读取。
assert not fixed  # 验证模型猜测不能作为长期事实消费。